# IFC Keyword Sensitivity Analysis

**Methodology step:** Step 11 — Apply feature tagging

This notebook compares two IFC keyword matching configurations to support the choice of `case_sensitive=True` in `create_kw_index.py`:

- **Run 1 (production):** `case_sensitive=True` — matches only `IFC` and `International Finance Corporation` in exact case, minimising false positives from common abbreviations.
- **Run 2 (alternative):** `case_sensitive=False` — matches all case variants; used to quantify the sensitivity of IFC-tag counts to this design choice.

The notebook identifies projects whose IFC-tag status changes between the two runs, extracts matched snippets for manual review, and reports the magnitude of the difference.

In [1]:
import json
import paths
import pandas as pd
from keyword_matcher import KeywordMatcher, compare_runs, extract_snippets

KEYWORDS_DIR = paths.DATA_ROOT / "reference" / "keywords"
IFC_KEYWORDS_PATH = KEYWORDS_DIR / "ifc_keywords.json"
CLEANED_DATA_PATH = paths.DATA_ROOT / "intermediate" / "cleaned_dataset.xlsx"

# Load IFC keywords once — reused for both matching runs and extract_snippets
# See matcher/examples/compare_keyword_runs.py for the reusable template for this workflow.
with open(IFC_KEYWORDS_PATH) as f:
    ifc_kw_dict = json.load(f)

ifc_keywords = [kw for kws in ifc_kw_dict.values() for kw in kws]

In [2]:
df = pd.read_excel(CLEANED_DATA_PATH)
print(df.shape)
df.head(2)

(7574, 29)


,project_id,project_name,wb_region,country,country_lending_group,fcs_status,practice_group,global_practice,wb_agreement_type,wb_lending_instrument_type,...,ieg_monitoring_and_evaluation_quality_ratings,text_url,pdf_url,lessons,ieg_outcome_ratings_num,ieg_bank_performance_ratings_num,ieg_quality_at_entry_ratings_num,ieg_quality_of_supervision_ratings_num,cohort,decade
0,P000001,West Africa Pilot Community-based Natural Reso...,Other,Africa,NaN,Non-FCS,Sustainable Development,Environment,GEF,IPF,...,NaN,http://documents.worldbank.org/curated/en/6526...,http://documents.worldbank.org/curated/en/6526...,The ICR contains an excellent set of thoughtfu...,2.0,2.0,2.0,2.0,2003-2007,1998-2007
1,P000003,REIMP(CEN.ENV.INFO),Other,Africa,NaN,Non-FCS,Sustainable Development,Environment,GEF,IPF,...,NaN,http://documents.worldbank.org/curated/en/4706...,http://documents.worldbank.org/curated/en/4706...,1. Even in a region beset by poverty and civil...,5.0,5.0,6.0,5.0,2003-2007,1998-2007


In [7]:
(
    df
    .assign(
        app_to_eval = lambda x: x.evaluation_fy - x.approval_fy,
        app_to_closing = lambda x: x.closing_fy - x.approval_fy,
        closing_to_eval = lambda x: x.evaluation_fy - x.closing_fy
    )
    .filter(['app_to_eval','app_to_closing','closing_to_eval'])
    .rename(columns={
        'app_to_eval':'Approval to Evaluation (FYs)',
        'app_to_closing':'Approval to Closing (FYs)',
        'closing_to_eval':'Closing to Evaluation (FYs)'
    })
    .mean()
)

Approval to Evaluation (FYs)    7.528262
Approval to Closing (FYs)       5.836768
Closing to Evaluation (FYs)     1.618503
dtype: float64

## Run 1: case_sensitive=True (production config)

This matches `create_kw_index.py` exactly: substring match, case-sensitive, so only `IFC` and `International Finance Corporation` (exact case) are matched.

In [11]:
df_sensitive = (
    KeywordMatcher(df=df, text_column="lessons", copy=True)
    .add_dictionary(
        ifc_kw_dict,
        dict_name="ifc",
        flatten=False,
        output_format="binary",
        match_mode="substring",
        case_sensitive=True,
    )
    .process()
)

print(f"Projects tagged (case-sensitive):   {df_sensitive['ifc_tag'].sum()}")

Projects tagged (case-sensitive):   59


## Run 2: case_sensitive=False

Same config but case-insensitive — will also match lowercase `ifc`.

In [12]:
df_insensitive = (
    KeywordMatcher(df=df, text_column="lessons", copy=True)
    .add_dictionary(
        ifc_kw_dict,
        dict_name="ifc",
        flatten=False,
        output_format="binary",
        match_mode="substring",
        case_sensitive=False,
    )
    .process()
)

print(f"Projects tagged (case-insensitive): {df_insensitive['ifc_tag'].sum()}")

Projects tagged (case-insensitive): 70


## Compare the two runs

In [13]:
# df_sensitive = baseline (production), df_insensitive = proposed change
comparison = compare_runs(
    df_before=df_sensitive,
    df_after=df_insensitive,
    join_key="project_id",
    columns=["ifc_tag"],
)

comparison.summary

,column,n_changed,n_increased,n_decreased,n_unchanged
0,ifc_tag,11,11,0,7563


In [14]:
comparison.changes

ifc_tag       
             after before
project_id               
P005836        1.0    0.0
P005907        1.0    0.0
P008327        1.0    0.0
P040142        1.0    0.0
P040830        1.0    0.0
P043367        1.0    0.0
P048026        1.0    0.0
P049582        1.0    0.0
P049718        1.0    0.0
P145488        1.0    0.0
P150194        1.0    0.0

In [15]:
# Projects that gained the IFC tag when switching to case-insensitive
gained_ids = comparison.changes["ifc_tag"].query("after > before").index
gained = df.set_index("project_id").loc[gained_ids, ["lessons"]].reset_index()

print(f"{len(gained)} project(s) gained the ifc_tag")
gained[["project_id"]]

11 project(s) gained the ifc_tag


,project_id
0,P005836
1,P005907
2,P008327
3,P040142
4,P040830
5,P043367
6,P048026
7,P049582
8,P049718
9,P145488


## Inspect what matched in the newly tagged projects

In [16]:
from IPython.display import display, Markdown

for _, row in gained.iterrows():
    lines = [f"**=== {row['project_id']} ===**"]
    snippets = extract_snippets(row["lessons"], ifc_keywords, context=60, case_sensitive=False, highlight=True)
    for s in snippets:
        lines.append(f"&nbsp;&nbsp;[{s['keyword']}] ...{s['snippet']}...")
    display(Markdown("  \n".join(lines)))

**=== P005836 ===**  
&nbsp;&nbsp;[IFC] ...(FY03) i s promoting the urbancorporate reform agenda, and **ifc**oupled with a greater WRM focus, can make a major difference...

**=== P005907 ===**  
&nbsp;&nbsp;[IFC] ...(FY03) i s promoting the urbancorporate reform agenda, and **ifc**oupled with a greater WRM focus, can make a major difference...

**=== P008327 ===**  
&nbsp;&nbsp;[IFC] ...hofcapitalmarkets. e Mass voucherprivatization seldomworks, **ifc**hecks andbalanceson corporate governanceare not inplace andf...

**=== P040142 ===**  
&nbsp;&nbsp;[IFC] ...hofcapitalmarkets. e Mass voucherprivatization seldomworks, **ifc**hecks andbalanceson corporate governanceare not inplace andf...

**=== P040830 ===**  
&nbsp;&nbsp;[IFC] ...hofcapitalmarkets. e Mass voucherprivatization seldomworks, **ifc**hecks andbalanceson corporate governanceare not inplace andf...

...

**=== P048026 ===**  
&nbsp;&nbsp;[IFC] ...suffers because objectives are left alone but design is sign**ifc**antly adjusted....

**=== P049582 ===**  
&nbsp;&nbsp;[IFC] ...ighquality hands-on technical advice on solutions as needed.**Ifc**urrent revenues are insufficient andrepeatedrequests for str...

**=== P049718 ===**  
&nbsp;&nbsp;[IFC] ...ters needs to be accompanied by a public awareness campaign **ifc**hanges inthe rules o f the game are going to leadto water co...

**=== P145488 ===**  
&nbsp;&nbsp;[IFC] ...CR that flexibility should be built into the design of DPOs **ifc**ircumstances change, as they did when Tuvalus revenue stream...

**=== P150194 ===**  
&nbsp;&nbsp;[IFC] ...CR that flexibility should be built into the design of DPOs **ifc**ircumstances change, as they did when Tuvalus revenue stream...